In [ ]:
!git clone https://github.com/YPolina/Medicine.git

In [ ]:
%cd ./Medicine/BELKA/training

In [ ]:
!pip install -r ../requirements.txt

In [ ]:
import sys
import h5py
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd

sys.path.append(os.path.abspath(os.path.join('..')))

sys.modules.pop("functionality.models", None)
sys.modules.pop("functionality.data_preparation", None)
from functionality.data_preparation import CNN_Dataset, train_model
from functionality.models import CNNBinaryClassifierLightning

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger
from transformers import AutoTokenizer, AutoModel

import torch
import pickle
from tqdm import tqdm
import numpy as np
import gc
from torch.cuda.amp import autocast

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def compute_and_save_encoded_smiles(smiles, labels, save_path, batch_size=1000, max_length=142):
    """
    Compute encoded SMILES representations in batches and save them efficiently using HDF5

    Args:
        smiles (pd.Series): Data containing SMILES strings
        labels (pd.Series): Corresponding labels
        save_path (str): Path to save computed encoded SMILES and labels
        batch_size (int): Number of SMILES strings to process per batch
        max_length (int): Maximum length of encoded SMILES strings
    """
    # Encoding dictionary
    enc_dict = {
        'l': 1, 'y': 2, '@': 3, '3': 4, 'H': 5, 'S': 6, 'F': 7, 'C': 8, 'r': 9, 's': 10, '/': 11, 'c': 12, 'o': 13,
        '+': 14, 'I': 15, '5': 16, '(': 17, '2': 18, ')': 19, '9': 20, 'i': 21, '#': 22, '6': 23, '8': 24, '4': 25, '=': 26,
        '1': 27, 'O': 28, '[': 29, 'D': 30, 'B': 31, ']': 32, 'N': 33, '7': 34, 'n': 35, '-': 36
    }

    def encode_smile(smile):
        """Encodes a SMILES string to numerical values and pads it"""
        encoded = [enc_dict.get(char, 0) for char in smile]
        padded = encoded + [0] * (max_length - len(encoded))
        return padded

    with h5py.File(save_path, "w") as h5f:
        # Create datasets for encoded SMILES and labels
        dset_encoded_smiles = h5f.create_dataset(
            "encoded_smiles", 
            shape=(0, max_length), 
            maxshape=(None, max_length), 
            dtype=np.int64, 
            compression="gzip"
        )
        dset_labels = h5f.create_dataset(
            "labels", 
            shape=(0,), 
            maxshape=(None,), 
            dtype=np.int64, 
            compression="gzip"
        )

        for i in tqdm(range(0, len(smiles), batch_size), desc="Encoding SMILES"):
            batch_smiles = smiles.iloc[i : i + batch_size].tolist()
            batch_labels = labels.iloc[i : i + batch_size].values.astype(np.int64)

            batch_encoded_smiles = np.array([encode_smile(smile) for smile in batch_smiles], dtype=np.int64)

            #Resize datasets and append new data
            dset_encoded_smiles.resize(dset_encoded_smiles.shape[0] + batch_encoded_smiles.shape[0], axis=0)
            dset_encoded_smiles[-batch_encoded_smiles.shape[0]:] = batch_encoded_smiles

            dset_labels.resize(dset_labels.shape[0] + batch_labels.shape[0], axis=0)
            dset_labels[-batch_labels.shape[0]:] = batch_labels

            #Memory cleanup
            del batch_smiles, batch_labels, batch_encoded_smiles
            gc.collect()

    print(f"Encoded SMILES and labels saved to {save_path}")
    return save_path


In [ ]:
protein_names = ['sEH', 'BRD4', 'HSA']
save_dir = '/content/drive/MyDrive/embeddings/'
model_name = 'CNN'
batch_size=1000

for protein_name in protein_names:
    print(f"Smiles encoding for protein: {protein_name}")

    train_data = pd.read_parquet(f'../intermediates/train_data/{protein_name}/{protein_name}_train.parquet')
    val_data = pd.read_parquet(f'../intermediates/train_data/{protein_name}/{protein_name}_val.parquet')

    train_embeddings_path = os.path.join(save_dir, f"{protein_name}_{model_name}_train.h5")
    val_embeddings_path = os.path.join(save_dir, f"{protein_name}_{model_name}_val.h5")

    compute_and_save_encoded_smiles(train_data["molecule_smiles"], train_data['binds'])
    compute_and_save_encoded_smiles(val_data["molecule_smiles"], val_data['binds'])
